In [ ]:
/*
The plan

coordinator agent that calls:
- flight agent (to find flights for the wedding party)
    - We used the kiwi MCP for this, can try this again
    - What about flightpath? It is used by humans to book agents
    - There's something similar for hotels, like amadeus or odyssey
    - There's RouteStack.ai, which combines flight, hotels, car rentals, and activities in a single MCP
- music playlist agent (to plan music for throughout the day based on the happy couple's preferences)
    - Can use Tavily for what's most popular right now
    - Older music can probably be found from the agent itself, though niche choices may not work, so ability to search the web should be available
- venue agent (to find venues within the budget and in the area you would like to have the wedding)
    - Can use Tavily to search venues online


Remember, each agent is focusing on its own specific task, and is making its own specific tool calls. The coordinator agent, calls the subagents based on the user's query - this could be query for just 1 section, e.g. venue, or multiple/all sections.
    
Approach:
- start with simple invoke function calls with queries based on that. Extend it to read from files (e.g. imagine you've been filling in a speadsheet/got data from your wedding RSVP website, and you want to pass that to an agent). This may be useful for determining the playlist, the flights, or even the venue (e.g. accessibility requirements). 
- start with the agent outputting simple text. Extend it to write to a spreadsheet.
- I saw that handling failures was discussed in the solution - have a think and implement something!

*/

In [17]:
from dotenv import load_dotenv

load_dotenv()


True

In [18]:
# define MCP
# let's start with kiwi, as we already have this, and maybe extend to using RouteStack.ai, to include hotels, cars, and more - in fact, this may be a simpler solution all around, but let's see how this progresses.
from langchain_mcp_adapters.client import MultiServerMCPClient # this is a visually simpler option that defining a single client, see docs: https://reference.langchain.com/python/langchain-mcp-adapters
from langchain_mcp_adapters.tools import load_mcp_tools # this will allow us to load tools individually


# When you call client.get_tools(), the client has to discover what tools each server offers, so it connects to every configured server (math and weather) and does the MCP handshake plus a tools/list request. That means the math subprocess is spawned and the weather HTTP endpoint is contacted at that point. For the weather server, this is why it needs to be running on port 8000 before you call get_tools(); otherwise the call fails.
# What it doesn’t do is keep those sessions open. Discovery uses a short-lived session per server, which is closed once the tool list is returned. So after get_tools() returns, the math subprocess has exited, and nothing is held open.
client = MultiServerMCPClient(
    {
        "travel_server": {
            "transport": "streamable_http",
            "url": "https://mcp.kiwi.com"
        }
    }
)

mcp_tools = await client.get_tools()
print(len(mcp_tools))
for t in mcp_tools:
    print(t.name)
    print(t.description)
    print(t.args_schema)   # may be a dict (JSON schema) rather than a pydantic model
    print("-----")

2
search-flight
# Search for flights

Searches Kiwi.com for available flights between two locations for the given dates and passengers. City or airport names are resolved automatically, so call this whenever the user wants to search for flights — whether they gave IATA codes or just place names.

## Result shape

Returns `{ query, currency, passengers, resultsCount, itineraries, searchTimeMs }`. Each item in `itineraries` has:
- `price` (number) and `priceFormatted` (e.g. "123 EUR")
- `totalDurationSeconds`
- `bookingUrl` — the link to book the flight
- `imageId` — destination city id for a hero photo (https://images.kiwi.com/photos/600x600/{imageId}.jpg)
- `baggage` — total included baggage across all travelers: `{ personalItem, cabinBag, checkedBag }` (counts)
- `outbound` (and `inbound` for return flights), each a leg with: `route` (list of airport codes including layovers, e.g. ["PRG","MAD","BCN"]), `departureTime` / `arrivalTime` (local ISO timestamps), `durationSeconds`, `stops`,

In [25]:
# remember, subagents are basically tools - yay!
from langchain.tools import tool
from langchain.messages import HumanMessage
from langchain.agents import create_agent
from pydantic import BaseModel
from langgraph.checkpoint.memory import InMemorySaver

### Travel Agent
class TravelInfo(BaseModel):
    dates: str = Field(description="The dates of travel, e.g. 1st to the 15th of April, 18/08/2027-25/08/2027, start to end of January")
    current_location: str = Field(description="The current location of the traveller, e.g. Weston-super-Mare, LDN, UK")
    destination: str = Field(description="The destination that the traveller wants to reach, e.g. Atlanta, LA, USA")
    travel_class: str = Field(description="The class of travel, e.g. 1st, economy")

travel_agent = create_agent(
    model='ollama:gemma4',
    tools=mcp_tools,
    checkpointer=InMemorySaver()
)

# user_query = "I live in London and I am travelling to Atlana. I want to leave on the 1st of March, and arrive back on the 15th of March. I want to fly in 1st class. I am travelling in 2027. I am traveling alone. I am over 18 years old."
# response = await travel_agent.ainvoke({"messages": [HumanMessage(content=user_query)]})

# print(response)

config = {"configurable": {"thread_id": "1"}}
# args_schema inforces the expected input for the tool, so the main agent would need to pass the info in the TravelInfo schema
# we then define it as an object and unpack the **kwargs in the subagent invocation
@tool(args_schema=TravelInfo)
async def call_travel_agent(**kwargs) -> str:
    """Call travel agent to get flights based on the user's query"""
    travel_info = TravelInfo(**kwargs)
    response = await travel_agent.ainvoke({"messages": [HumanMessage(
        content=f"Find flights from {travel_info.current_location} to {travel_info.destination} in {travel_info.travel_class}, between the following dates: {travel_info.dates}."
    )]}, config)
    return response["messages"][-1].content

In [32]:
### Main agent

### A note of async use due to the MCP server
# The subagent has to be async because of the MCP tools. They’re async-only, so travel_agent must be run with await travel_agent.ainvoke(...). A sync invoke would hit a tool with no sync implementation and fail.
# The main agent has to be async because of your wrapper tool. Once call_travel_agent is async def, it only has a coroutine implementation, so the main agent has to run it with await main_agent.ainvoke(...). Calling main_agent.invoke(...) would fail for the same reason: no sync implementation to call.
# So it’s a chain: MCP tools are async, so the subagent call is async, so the wrapper tool is async, so the main agent call is async. The MCP requirement propagates upward through every layer.

main_agent = create_agent(
    model='ollama:gemma4',
    tools=[call_travel_agent]
)

user_query = "I live in London and I am travelling to Atlana. I want to leave on the 1st of March, and arrive back on the 15th of March. I want to fly in 1st class. I am travelling in 2027. I am traveling alone. I am over 18 years old."

main_agent_response = await main_agent.ainvoke({"messages": [HumanMessage(content=user_query)]})

print("just the main response:")
print(main_agent_response)

print("messages only:")
print(main_agent_response["messages"][-1].content)

just the main response:
{'messages': [HumanMessage(content='I live in London and I am travelling to Atlana. I want to leave on the 1st of March, and arrive back on the 15th of March. I want to fly in 1st class. I am travelling in 2027. I am traveling alone. I am over 18 years old.', additional_kwargs={}, response_metadata={}, id='02e72b46-f74e-401e-9872-9db25c722b5d'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4', 'created_at': '2026-10-21T20:31:23.594138Z', 'done': True, 'done_reason': 'stop', 'total_duration': 22469784208, 'load_duration': 2871541, 'prompt_eval_count': 179, 'prompt_eval_duration': 564141000, 'eval_count': 640, 'eval_duration': 21886401000, 'logprobs': None, 'model_name': 'gemma4', 'model_provider': 'ollama'}, id='lc_run--01a16028-fc02-7a32-8e96-25751241a005-0', tool_calls=[{'name': 'call_travel_agent', 'args': {'current_location': 'London', 'dates': '2027-03-01 to 2027-03-15', 'destination': 'Atlanta', 'travel_class': '1st class'},

In [26]:
from tavily import TavilyClient
tavily_client = TavilyClient()

In [27]:
# remember, subagents are basically tools - yay!
from langchain.tools import tool
from langchain.messages import HumanMessage
from langchain.agents import create_agent
from pydantic import BaseModel, Field
from typing import Any
from langgraph.checkpoint.memory import InMemorySaver

### Music Agent
class PlaylistInfo(BaseModel):
    era: str = Field(description="The time period of when the music was popular or was released, 1920s, 80s, noughties")
    genre: str = Field(description="The music genre, e.g. Pop, Hip-Hop, Rock, Metal, Hardcore, Dance, EDM")
    setting: str = Field(description="The period during which the music would be played, e.g. wedding ceremony, wedding breakfast, cocktail hour, after-party")

# define the web searching tool, Tavily offers other options beyond search, but we'll keep it as a search for now
@tool
def find_music(query: str) -> dict[str, Any]:
    """Search the web to find music based on the user's query
    
    Args:
        query: The type of music that the user would like to find.
    """
    return tavily_client.search(query)


system_prompt = """
    You are an agent designed to search for music based on the user's query. 
    
    Use the tools provided to search for music if you are being asked to respond based on dates you have not been trained on. 
    """
music_agent = create_agent(
    model='ollama:gemma4',
    tools=[find_music],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()
)

config = {"configurable": {"thread_id": "1"}}
# args_schema inforces the expected input for the tool, so the main agent would need to pass the info in the TravelInfo schema
# we then define it as an object and unpack the **kwargs in the subagent invocation
@tool(args_schema=PlaylistInfo)
def call_music_agent(**kwargs) -> str:
    """Call the music agent to find suitable music for the wedding based on the user's query"""
    playlist_info = PlaylistInfo(**kwargs)
    response = music_agent.invoke({"messages": [HumanMessage(
        content=f"Find {playlist_info.genre} songs from the {playlist_info.era} to be suitable during the {playlist_info.setting} of a wedding."
    )]}, config)
    for m in response["messages"]:
        print(type(m).__name__, getattr(m, "name", None), getattr(m, "tool_calls", None))

    # print everything to debug
    print(response)
    # we won't see a full trace of music_agent's tool calls in the main_agent's response!
    # so we need to add something like the above print statement to check tool calls in the music_agent
    return response["messages"][-1].content

In [53]:
### Main agent
main_agent = create_agent(
    model='ollama:gemma4',
    tools=[call_music_agent]
)

user_query = "I would like some 2026 hardcore punk at the wedding after party. I need something high energy!"

main_agent_response = main_agent.invoke({"messages": [HumanMessage(content=user_query)]})

print("just the main response:")
print(main_agent_response)

print("messages only:")
print(main_agent_response["messages"][-1].content)

HumanMessage None None
AIMessage None [{'name': 'find_music', 'args': {'query': 'hardcore punk songs suitable for a wedding after party'}, 'id': '0ebea97b-e122-4a3b-87de-2743f00ee124', 'type': 'tool_call'}]
ToolMessage find_music None
AIMessage None []
{'messages': [HumanMessage(content='Find hardcore punk songs from the 2026 to be suitable during the wedding after party of a wedding.', additional_kwargs={}, response_metadata={}, id='7bb6b43f-b025-4bf4-97b9-ab2b31620efa'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4', 'created_at': '2026-10-21T21:28:16.418188Z', 'done': True, 'done_reason': 'stop', 'total_duration': 10611450416, 'load_duration': 2479458, 'prompt_eval_count': 153, 'prompt_eval_duration': 470577000, 'eval_count': 300, 'eval_duration': 10124708000, 'logprobs': None, 'model_name': 'gemma4', 'model_provider': 'ollama'}, id='lc_run--01a1605d-3dad-7d62-b24b-aea3eb3918f3-0', tool_calls=[{'name': 'find_music', 'args': {'query': 'hardcore punk

In [28]:
# the venue agent
# remember, subagents are basically tools - yay!
from langchain.tools import tool
from langchain.messages import HumanMessage
from langchain.agents import create_agent
from pydantic import BaseModel, Field
from typing import Any
from langgraph.checkpoint.memory import InMemorySaver

### Venue Agent
class VenueInfo(BaseModel):
    location: str = Field(description="Geographical location, e.g. Cornwall, UK, European Union")
    features: list[str] = Field(description="Features that the wedding venue must have, e.g. beautiful garden, historic building, beach")
    price: str = Field(description="Venue hire cost, e.g. £10,000, cheap, ten thousand pounds")

# define the web searching tool, Tavily offers other options beyond search, but we'll keep it as a search for now
@tool
def find_venue(query: str) -> dict[str, Any]:
    """Search the web to find a wedding venue based on the user's query
    
    Args:
        query: The type of venue that the user would like to find.
    """
    return tavily_client.search(query)


system_prompt = """
    You are an agent designed to search for wedding venues based on the user's query. 
    
    Use the tools provided to search for a wedding venue. 
    """
venue_agent = create_agent(
    model='ollama:gemma4',
    tools=[find_venue],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()
)

config = {"configurable": {"thread_id": "1"}}
# args_schema inforces the expected input for the tool, so the main agent would need to pass the info in the TravelInfo schema
# we then define it as an object and unpack the **kwargs in the subagent invocation
@tool(args_schema=VenueInfo)
def call_venue_agent(**kwargs) -> str:
    """Call the venue agent to find suitable wedding venue based on the user's query"""
    venue_info = VenueInfo(**kwargs)
    response = venue_agent.invoke({"messages": [HumanMessage(
        content=f"Find a wedding venue that is located in {venue_info.location}. The venue must have the following features: {venue_info.features}. The venue cost should be around {venue_info.price}."
    )]}, config)
    for m in response["messages"]:
        print(type(m).__name__, getattr(m, "name", None), getattr(m, "tool_calls", None))

    # print everything to debug
    print(response)
    # we won't see a full trace of venue_agent's tool calls in the main_agent's response!
    # so we need to add something like the above print statement to check tool calls in the venue_agent
    return response["messages"][-1].content

In [9]:
### Main agent
main_agent = create_agent(
    model='ollama:gemma4',
    tools=[call_venue_agent],
    checkpointer=InMemorySaver()
)

user_query = "I want to have a wedding on a beach in cornwall. My budget is 15 thousand pounds."

main_agent_response = main_agent.invoke({"messages": [HumanMessage(content=user_query)]})

print("just the main response:")
print(main_agent_response)

print("messages only:")
print(main_agent_response["messages"][-1].content)

HumanMessage None None
AIMessage None [{'name': 'find_venue', 'args': {'query': 'wedding venue in Cornwall with a beach and a budget of 15,000 pounds'}, 'id': '1470769f-db79-496c-a65e-a5ad27b814c1', 'type': 'tool_call'}]
ToolMessage find_venue None
AIMessage None []
{'messages': [HumanMessage(content="Find a wedding venue that is located in Cornwall. The venue must have the following features: ['beach']. The venue cost should be around 15 thousand pounds.", additional_kwargs={}, response_metadata={}, id='4e7f0209-ae7c-48c2-9443-f5c43accf74d'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4', 'created_at': '2026-10-22T21:26:09.871358Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3570131625, 'load_duration': 3339084, 'prompt_eval_count': 152, 'prompt_eval_duration': 469368000, 'eval_count': 93, 'eval_duration': 3074965000, 'logprobs': None, 'model_name': 'gemma4', 'model_provider': 'ollama'}, id='lc_run--01a16581-c6db-7713-b4e7-314b596e5518-0'

In [29]:
from langgraph.checkpoint.memory import InMemorySaver

### Main agent calling everything!
main_agent = create_agent(
    model='ollama:gemma4',
    tools=[call_venue_agent, call_music_agent, call_travel_agent],
    checkpointer=InMemorySaver()
)

user_query = """
    I want to have a wedding on a beach in cornwall. My budget is 15 thousand pounds.

    During the wedding after-party, I would like some Hip-Hop and Rock that came out in 2026. I need something high energy!

    And I will need to fly my wedding party from Bristol to the location you recommend in Cornwall. Help me find suitable flights in economy class for 6 adults, in the June. 
    """

config = {"configurable": {"thread_id": "1"}}
main_agent_response = main_agent.invoke({"messages": [HumanMessage(content=user_query)]}, config)

print("just the main response:")
print(main_agent_response)

print("messages only:")
print(main_agent_response["messages"][-1].content)

HumanMessage None None
AIMessage None [{'name': 'find_venue', 'args': {'query': 'wedding venue in Cornwall with a beach and a budget of around £15000'}, 'id': '0fa3dddc-557f-4722-a17d-a218b34eaaf3', 'type': 'tool_call'}]
ToolMessage find_venue None
AIMessage None []
{'messages': [HumanMessage(content="Find a wedding venue that is located in Cornwall. The venue must have the following features: ['beach']. The venue cost should be around 15000 pounds.", additional_kwargs={}, response_metadata={}, id='d3cef03b-9acd-4cf9-8f62-57e0ac3afe29'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4', 'created_at': '2026-10-22T21:55:20.529893Z', 'done': True, 'done_reason': 'stop', 'total_duration': 7204816959, 'load_duration': 965542, 'prompt_eval_count': 154, 'prompt_eval_duration': 474658000, 'eval_count': 200, 'eval_duration': 6721944000, 'logprobs': None, 'model_name': 'gemma4', 'model_provider': 'ollama'}, id='lc_run--01a1659c-6f2c-7753-a22e-db0a90df241f-0', tool

NotImplementedError: StructuredTool does not support sync invocation.

In [ ]:
# Notes to self:
# how do we do continuous conversations with the agent? Is it just a case of enabling memory, and prompting the agent again?
# maybe I can ask the main agent to ask me questions upfront to clarify my prompt before it goes to call subagents? I can include this guidance in its system prompt, like "ask the user clarifying questions before caling tools"